# 2. 正负通道分离

将有符号 sEMG 拆分为 positive/negative 两个非负通道（`x = positive - negative` 可精确重建）。

In [1]:
import hashlib
import json
import os
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import torch
from tqdm.auto import tqdm


def find_project_root():
    # 允许从项目根目录或 notebook 所在目录启动 Jupyter。
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src' / 'data_prepare').is_dir() and (candidate / 'CapgMyo_data').is_dir():
            return candidate
    raise RuntimeError('找不到项目根目录：应同时包含 src/data_prepare 和 CapgMyo_data。')


PROJECT_ROOT = find_project_root()
RAW_ROOT = PROJECT_ROOT / 'CapgMyo_data' / 'raw'
OUTPUT_ROOT = PROJECT_ROOT / 'CapgMyo_data' / 'raw_polarity'
SOURCE_MANIFEST_PATH = RAW_ROOT / 'split_manifest.json'
OUTPUT_MANIFEST_PATH = OUTPUT_ROOT / 'manifest.json'
SPLIT_NAMES = ('train', 'val', 'test')
SOURCE_DIRS = {name: RAW_ROOT / name for name in SPLIT_NAMES}
OUTPUT_DIRS = {name: OUTPUT_ROOT / name for name in SPLIT_NAMES}
EXPECTED_COUNTS = {'train': 1008, 'val': 144, 'test': 288}
EXPECTED_INPUT_SHAPE = (1000, 1, 8, 16)
EXPECTED_OUTPUT_SHAPE = (1000, 2, 8, 16)
SPLIT_REPETITIONS = {
    'train': {1, 3, 4, 5, 6, 7, 8},
    'val': {10},
    'test': {2, 9},
}
OVERWRITE = False

for directory in (OUTPUT_ROOT, *OUTPUT_DIRS.values()):
    directory.mkdir(parents=True, exist_ok=True)

print(f'输入目录：{RAW_ROOT}')
print(f'输出目录：{OUTPUT_ROOT}')
print(f'覆盖已有文件：{OVERWRITE}')

输入目录：/root/autodl-tmp/Neurophic_System2CapgMyo/CapgMyo_data/raw
输出目录：/root/autodl-tmp/Neurophic_System2CapgMyo/CapgMyo_data/raw_polarity
覆盖已有文件：False


In [2]:
METADATA_KEYS = ('label', 'subject_id', 'gesture_id', 'repetition_id')
FILE_NAME_PATTERN = re.compile(r's(\d+)_g(\d+)_r(\d+)\.pt$', re.IGNORECASE)


def load_pt(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        # 兼容尚未提供 weights_only 参数的旧版 PyTorch。
        return torch.load(path, map_location='cpu')


def write_json_atomic(path, value):
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    temporary_path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    os.replace(temporary_path, path)


def sha256sum(path):
    digest = hashlib.sha256()
    with path.open('rb') as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def scalar_int(value, key, path):
    if not isinstance(value, torch.Tensor) or value.ndim != 0 or value.dtype != torch.int64:
        raise TypeError(f'{path} 的 {key} 必须是 torch.int64 标量。')
    return int(value)


def validate_metadata(payload, path):
    missing = {'data', *METADATA_KEYS} - set(payload)
    if missing:
        raise KeyError(f'{path} 缺少字段：{sorted(missing)}')

    metadata = {key: scalar_int(payload[key], key, path) for key in METADATA_KEYS}
    if metadata['label'] != metadata['gesture_id'] - 1:
        raise ValueError(f'{path} 的 label 与 gesture_id 不一致。')
    if not 0 <= metadata['label'] <= 7:
        raise ValueError(f"{path} 的 label 越界：{metadata['label']}")
    if not 1 <= metadata['subject_id'] <= 18:
        raise ValueError(f"{path} 的 subject_id 越界：{metadata['subject_id']}")
    if not 1 <= metadata['gesture_id'] <= 8:
        raise ValueError(f"{path} 的 gesture_id 越界：{metadata['gesture_id']}")
    if not 1 <= metadata['repetition_id'] <= 10:
        raise ValueError(f"{path} 的 repetition_id 越界：{metadata['repetition_id']}")

    match = FILE_NAME_PATTERN.fullmatch(path.name)
    if match is None:
        raise ValueError(f'文件名不符合 sXX_gXX_rXX.pt：{path.name}')
    file_subject, file_gesture, file_repetition = map(int, match.groups())
    if (file_subject, file_gesture, file_repetition) != (
        metadata['subject_id'], metadata['gesture_id'], metadata['repetition_id']
    ):
        raise ValueError(f'{path} 的文件名编号与内部元数据不一致。')
    return metadata


def validate_raw_payload(payload, path):
    if not isinstance(payload, dict):
        raise TypeError(f'{path} 的内容必须是字典。')
    metadata = validate_metadata(payload, path)
    data = payload['data']
    if not isinstance(data, torch.Tensor):
        raise TypeError(f'{path} 的 data 必须是 Tensor。')
    if tuple(data.shape) != EXPECTED_INPUT_SHAPE or data.dtype != torch.float32:
        raise ValueError(
            f'{path} 的 data 为 shape={tuple(data.shape)}, dtype={data.dtype}；'
            f'期望 shape={EXPECTED_INPUT_SHAPE}, dtype=torch.float32。'
        )
    if not torch.isfinite(data).all():
        raise ValueError(f'{path} 的 data 包含 NaN 或 Inf。')
    return data, metadata


def encode_polarity(data):
    positive = data.clamp_min(0)
    negative = (-data).clamp_min(0)
    return torch.cat((positive, negative), dim=1).contiguous()


def validate_encoded_payload(payload, raw_data, metadata, path):
    if not isinstance(payload, dict):
        raise TypeError(f'{path} 的内容必须是字典。')
    encoded_metadata = validate_metadata(payload, path)
    if encoded_metadata != metadata:
        raise ValueError(f'{path} 的元数据与原始样本不一致。')

    data = payload['data']
    if not isinstance(data, torch.Tensor):
        raise TypeError(f'{path} 的 data 必须是 Tensor。')
    if tuple(data.shape) != EXPECTED_OUTPUT_SHAPE or data.dtype != torch.float32:
        raise ValueError(
            f'{path} 的 data 为 shape={tuple(data.shape)}, dtype={data.dtype}；'
            f'期望 shape={EXPECTED_OUTPUT_SHAPE}, dtype=torch.float32。'
        )
    if not torch.isfinite(data).all() or torch.any(data < 0):
        raise ValueError(f'{path} 的正负通道必须为有限的非负值。')

    positive = data[:, 0:1]
    negative = data[:, 1:2]
    if torch.any((positive != 0) & (negative != 0)):
        raise ValueError(f'{path} 同一位置的正负通道不能同时非零。')
    if not torch.equal(positive - negative, raw_data):
        raise ValueError(f'{path} 无法精确重建原始有符号信号。')
    return data


def output_is_valid(path, raw_data, metadata):
    if not path.is_file():
        return False
    try:
        validate_encoded_payload(load_pt(path), raw_data, metadata, path)
        return True
    except (EOFError, KeyError, OSError, RuntimeError, TypeError, ValueError):
        return False


def save_pt_atomic(path, payload):
    # 写入完成后再原子替换，避免中断产生半个可见的 .pt 文件。
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary_path)
    os.replace(temporary_path, path)

In [3]:
if not SOURCE_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f'缺少原始数据划分清单：{SOURCE_MANIFEST_PATH}')
source_manifest = json.loads(SOURCE_MANIFEST_PATH.read_text(encoding='utf-8'))
source_manifest_counts = source_manifest.get('split', {}).get('counts')
if source_manifest_counts != EXPECTED_COUNTS:
    raise RuntimeError(
        f'原始划分清单中的数量为 {source_manifest_counts}，期望 {EXPECTED_COUNTS}。'
    )

source_files = {}
for split_name in SPLIT_NAMES:
    files = sorted(SOURCE_DIRS[split_name].glob('subject_*/*.pt'))
    if len(files) != EXPECTED_COUNTS[split_name]:
        raise RuntimeError(
            f'{split_name} 中有 {len(files)} 个 .pt，期望 {EXPECTED_COUNTS[split_name]}。'
        )
    source_files[split_name] = files

print('原始数据文件数量检查通过：')
print({name: len(files) for name, files in source_files.items()})

原始数据文件数量检查通过：
{'train': 1008, 'val': 144, 'test': 288}


In [4]:
records = []
conversion_stats = Counter()
total_trials = sum(EXPECTED_COUNTS.values())

with tqdm(total=total_trials, unit='trial', desc='正负通道分离') as progress:
    for split_name in SPLIT_NAMES:
        for source_path in source_files[split_name]:
            raw_data, metadata = validate_raw_payload(load_pt(source_path), source_path)
            if metadata['repetition_id'] not in SPLIT_REPETITIONS[split_name]:
                raise ValueError(
                    f"{source_path} 的 repetition_id={metadata['repetition_id']} "
                    f'不属于 {split_name}。'
                )

            relative_path = source_path.relative_to(SOURCE_DIRS[split_name])
            output_path = OUTPUT_DIRS[split_name] / relative_path
            output_path.parent.mkdir(parents=True, exist_ok=True)

            if not OVERWRITE and output_is_valid(output_path, raw_data, metadata):
                conversion_stats['reused'] += 1
            else:
                encoded_data = encode_polarity(raw_data)
                output_payload = {
                    'data': encoded_data,
                    **{
                        key: torch.tensor(metadata[key], dtype=torch.int64)
                        for key in METADATA_KEYS
                    },
                }
                validate_encoded_payload(output_payload, raw_data, metadata, output_path)
                save_pt_atomic(output_path, output_payload)
                conversion_stats['written'] += 1

            records.append({
                'source_path': source_path.relative_to(PROJECT_ROOT).as_posix(),
                'output_path': output_path.relative_to(PROJECT_ROOT).as_posix(),
                'split': split_name,
                **metadata,
                'data_shape': list(EXPECTED_OUTPUT_SHAPE),
                'data_dtype': 'torch.float32',
            })
            progress.update(1)
            progress.set_postfix(split=split_name, subject=metadata['subject_id'])

print(f'转换完成：{dict(conversion_stats)}')

正负通道分离:   0%|          | 0/1440 [00:00<?, ?trial/s]

转换完成：{'written': 1440}


In [5]:
observed_counts = {
    split_name: len(list(OUTPUT_DIRS[split_name].glob('subject_*/*.pt')))
    for split_name in SPLIT_NAMES
}
if observed_counts != EXPECTED_COUNTS:
    raise RuntimeError(f'输出文件数量为 {observed_counts}，期望 {EXPECTED_COUNTS}。')

with tqdm(total=len(records), unit='trial', desc='校验正负通道') as progress:
    for record in records:
        source_path = PROJECT_ROOT / record['source_path']
        output_path = PROJECT_ROOT / record['output_path']
        raw_data, metadata = validate_raw_payload(load_pt(source_path), source_path)
        validate_encoded_payload(load_pt(output_path), raw_data, metadata, output_path)
        progress.update(1)

manifest = {
    'format_version': 1,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'dataset': 'CapgMyo DB-a polarity-separated official preprocessed data',
    'source': {
        'root': RAW_ROOT.relative_to(PROJECT_ROOT).as_posix(),
        'manifest': SOURCE_MANIFEST_PATH.relative_to(PROJECT_ROOT).as_posix(),
        'manifest_sha256': sha256sum(SOURCE_MANIFEST_PATH),
    },
    'output_root': OUTPUT_ROOT.relative_to(PROJECT_ROOT).as_posix(),
    'transformation': {
        'positive': 'max(x, 0)',
        'negative': 'max(-x, 0)',
        'channel_order': ['positive', 'negative'],
        'reconstruction': 'x = positive - negative',
        'normalization': None,
        'windowing': None,
        'spike_encoding': None,
    },
    'sample': {
        'input_shape': list(EXPECTED_INPUT_SHAPE),
        'output_shape': list(EXPECTED_OUTPUT_SHAPE),
        'data_dtype': 'torch.float32',
        'metadata_dtype': 'torch.int64',
    },
    'split_counts': observed_counts,
    'records': records,
}
write_json_atomic(OUTPUT_MANIFEST_PATH, manifest)

example_path = PROJECT_ROOT / records[0]['output_path']
example = load_pt(example_path)
print('全部校验通过。')
print(f'文件数量：{observed_counts}')
print(f'清单：{OUTPUT_MANIFEST_PATH}')
print(f'样例：{example_path.relative_to(PROJECT_ROOT)}')
print({
    'data_shape': tuple(example['data'].shape),
    'data_dtype': example['data'].dtype,
    'label': int(example['label']),
    'subject_id': int(example['subject_id']),
    'gesture_id': int(example['gesture_id']),
    'repetition_id': int(example['repetition_id']),
})

校验正负通道:   0%|          | 0/1440 [00:00<?, ?trial/s]

全部校验通过。
文件数量：{'train': 1008, 'val': 144, 'test': 288}
清单：/root/autodl-tmp/Neurophic_System2CapgMyo/CapgMyo_data/raw_polarity/manifest.json
样例：CapgMyo_data/raw_polarity/train/subject_01/s01_g01_r01.pt
{'data_shape': (1000, 2, 8, 16), 'data_dtype': torch.float32, 'label': 0, 'subject_id': 1, 'gesture_id': 1, 'repetition_id': 1}
